# Extension 3 SHAP: Full observed-demographic cohort

This notebook does not retrain any model. It loads the six final full-covariate checkpoints, uses 100 background households and all remaining 701 observed-demographic households for explanation, and runs both prediction heads for LSTM and Transformer seeds 7, 42, and 2024.

The 128 expected-gradient sample budget was selected by the prior 32-vs-64 convergence audit. The all-household sensitivity analysis remains a separate 100-household, seed-42 check because missing demographics are confounded with valid zero-coded categories.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys

WORKING = Path('/kaggle/working')
BUNDLE_MOUNT = Path('/kaggle/input/thesis-shap-full-run')
BUNDLE_ZIP = BUNDLE_MOUNT / 'thesis_shap_bundle.zip'
BUNDLE_DIR = BUNDLE_MOUNT / 'thesis-code'
INPUT_ROOT = Path('/kaggle/input')
PROJECT = WORKING / 'thesis-code'

assert BUNDLE_ZIP.exists() or BUNDLE_DIR.exists(), f'Missing bundle dataset: {BUNDLE_MOUNT}'
transaction_candidates = list(INPUT_ROOT.rglob('transaction_data.csv'))
assert transaction_candidates, 'Dunnhumby transaction_data.csv is not attached.'
DATA_MOUNT = transaction_candidates[0].parent
if PROJECT.exists():
    shutil.rmtree(PROJECT)
if BUNDLE_ZIP.exists():
    shutil.unpack_archive(str(BUNDLE_ZIP), str(WORKING))
else:
    shutil.copytree(BUNDLE_DIR, PROJECT)
os.chdir(PROJECT)

manifest = json.loads((PROJECT / 'BUNDLE_MANIFEST.json').read_text())
for row in manifest['files']:
    path = PROJECT / row['path']
    assert path.exists(), f'Missing bundled file: {path}'
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    assert digest == row['sha256'], f'Checksum mismatch: {path}'

raw_dir = PROJECT / 'data/raw/Dunnhumby datasets'
raw_dir.parent.mkdir(parents=True, exist_ok=True)
if raw_dir.exists() or raw_dir.is_symlink():
    raw_dir.unlink() if raw_dir.is_symlink() else shutil.rmtree(raw_dir)
raw_dir.symlink_to(DATA_MOUNT, target_is_directory=True)
required_data = [
    'transaction_data.csv', 'hh_demographic.csv', 'campaign_table.csv',
    'campaign_desc.csv', 'coupon_redempt.csv',
]
missing_data = [name for name in required_data if not (raw_dir / name).exists()]
assert not missing_data, f'Missing Dunnhumby files: {missing_data}'

gpu_name = subprocess.run(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
    check=True, capture_output=True, text=True,
).stdout.strip().splitlines()[0]
if 'P100' in gpu_name:
    print('P100 detected; installing the official CUDA 12.6 PyTorch wheel for Pascal support.')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
        '--force-reinstall', 'torch==2.9.1',
        '--index-url', 'https://download.pytorch.org/whl/cu126',
    ], check=True)

probe = subprocess.run([
    sys.executable, '-c',
    "import json, shap, torch; print(json.dumps({"
    "'cuda': torch.cuda.is_available(), 'torch': torch.__version__, "
    "'shap': shap.__version__, 'arches': torch.cuda.get_arch_list()}))",
], check=True, capture_output=True, text=True)
environment = json.loads(probe.stdout.strip().splitlines()[-1])
assert environment['cuda'], 'GPU is not enabled. Turn on a Kaggle GPU before running.'
assert 'sm_60' in environment['arches'] or 'P100' not in gpu_name, (
    f"Installed PyTorch build does not support the assigned P100: {environment['arches']}"
)
print('Bundle checksums verified:', len(manifest['files']), 'files')
print('GPU:', gpu_name)
print('Torch:', environment['torch'], '| SHAP:', environment['shap'])
print('Dunnhumby mount verified:', DATA_MOUNT)


In [ ]:
RESULTS_ROOT = WORKING / 'results/final_kaggle'
CHECKPOINT_ROOT = PROJECT / 'results/final_kaggle/checkpoints'
PILOT_CSV = PROJECT / 'pilot/shap_extension3_convergence.csv'

cmd = [
    sys.executable,
    'scripts/run_extension3_shap.py',
    '--results_root', str(RESULTS_ROOT),
    '--checkpoint_root', str(CHECKPOINT_ROOT),
    '--output_root', str(RESULTS_ROOT / 'shap'),
    '--device', 'cuda',
    '--analysis_seed', '42',
    '--n_background', '100',
    '--n_explain', '701',
    '--sensitivity_n_explain', '100',
    '--fixed_integration_samples', '128',
    '--pilot_convergence_csv', str(PILOT_CSV),
    '--n_bootstrap', '2000',
]
print('Starting full-cohort SHAP run. Expected Kaggle runtime: roughly 2-4 hours.')
print(' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import pandas as pd

tables = RESULTS_ROOT / 'tables'
summary = pd.read_csv(tables / 'shap_extension3_summary.csv')
additivity = pd.read_csv(tables / 'shap_extension3_additivity.csv')
run_manifest = json.loads((tables / 'shap_extension3_run_manifest.json').read_text())

assert len(summary) == 16, f'Expected 16 summary rows, found {len(summary)}'
assert set(summary['architecture']) == {'lstm', 'transformer'}
assert set(summary['head']) == {'freq', 'spend'}
assert set(summary['n_households']) == {701}
assert set(summary['n_seeds']) == {3}
assert set(summary['n_integration_samples']) == {128}
assert run_manifest['n_background'] == 100
assert run_manifest['n_explain'] == 701
assert run_manifest['sensitivity_n_explain'] == 100

display(summary.sort_values(
    ['architecture', 'head', 'relative_importance_pct'],
    ascending=[True, True, False],
))
display(additivity)
print('Maximum normalized additivity residual:', additivity['normalized_additivity_error'].max())
print('Interpretation: interventional model attribution conditional on transaction history, not causal campaign effects.')


In [ ]:
archive_base = WORKING / 'extension3_shap_full_cohort'
archive_path = shutil.make_archive(
    str(archive_base),
    'zip',
    root_dir=RESULTS_ROOT,
)

exports = {
    tables / 'shap_extension3_summary.csv': WORKING / 'shap_extension3_summary_full_cohort.csv',
    tables / 'shap_extension3_additivity.csv': WORKING / 'shap_extension3_additivity_full_cohort.csv',
    tables / 'shap_extension3_method_notes.md': WORKING / 'shap_extension3_method_notes_full_cohort.md',
    RESULTS_ROOT / 'plots/shap/shap_architecture_comparison.png': WORKING / 'shap_architecture_comparison_full_cohort.png',
}
for source, destination in exports.items():
    shutil.copy2(source, destination)

completion = WORKING / 'SHAP_FULL_COHORT_COMPLETE.txt'
completion.write_text(
    'Extension 3 SHAP full observed-demographic cohort completed successfully.\n'
    'Primary sample: 100 disjoint background + 701 explained households.\n'
    'Architectures: LSTM and Transformer; seeds: 7, 42, 2024; samples: 128.\n'
)
print('Complete. Download this Kaggle output:')
print(archive_path)
print('Archive size (MB):', round(Path(archive_path).stat().st_size / 1024**2, 1))
